In [12]:
import prospect
print(prospect.__version__)
import os
os.environ["SPS_HOME"] = "/home/hollman/fsps/"
os.environ["LD_LIBRARY_PATH"] = "/home/hollman/fsps/src:" + os.environ.get("LD_LIBRARY_PATH","")
import fsps
import dynesty
import sedpy
import h5py, astropy
import numpy as np
import astroquery
from astroquery.sdss import SDSS
from astropy.coordinates import SkyCoord
from astropy import units as u
from sedpy.observate import load_filters
from prospect.utils.obsutils import fix_obs
from prospect.models.templates import TemplateLibrary
from prospect.models import SpecModel
import prospect.sources.galaxy_basis as gb  # módulo donde vive CSPSpecBasis
gb.fsps = fsps
from prospect.sources import CSPSpecBasis
from prospect.fitting import lnprobfn, fit_model
from prospect.io import write_results as writer
from prospect.io import read_results as reader
import numpy as np
if not hasattr(np, "product"):
    np.product = np.prod
import matplotlib.pyplot as pl
from prospect.plotting import corner
from prospect.plotting.utils import best_sample
from prospect.plotting.utils import sample_posterior

1.4.0


In [10]:
from astropy.constants import c
import fsps
from prospect.models.templates import TemplateLibrary
from prospect.models import SpecModel
from prospect.models import priors
from prospect.sources import CSPSpecBasis, FastStepBasis

In [6]:
# Coordenadas del objeto (en grados) y creación de objeto SkyCoord
ra, dec = 200.947750, -1.547763
coord = SkyCoord(ra=ra*u.deg, dec=dec*u.deg)

# Consulta de espectros en SDSS dentro de ~2" de la posición dada
spectra = SDSS.get_spectra(coordinates=coord, radius=2*u.arcsec)
# Tomamos el primer espectro devuelto (formato astropy HDUList)
spec_hdulist = spectra[0]
spec_hdulist.info()  # Información sobre las extensiones HDU del FITS


Filename: (No file associated with this HDUList)
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     142   ()      
  1  COADD         1 BinTableHDU     26   3817R x 8C   [E, E, E, J, J, E, E, E]   
  2  SPECOBJ       1 BinTableHDU    262   1R x 126C   [6A, 4A, 16A, 23A, 16A, 8A, E, E, E, J, E, E, J, B, B, B, B, B, B, J, 22A, 19A, 19A, 22A, 19A, I, 3A, 3A, 1A, J, D, D, D, E, E, 19A, 8A, J, J, J, J, K, K, J, J, J, J, J, J, K, K, K, K, I, J, J, J, J, 5J, D, D, 6A, 21A, E, E, E, J, E, 24A, 10J, J, 10E, E, E, E, E, E, E, J, E, E, E, J, E, 5E, E, 10E, 10E, 10E, 5E, 5E, 5E, 5E, 5E, J, J, E, E, E, E, E, E, 25A, 21A, 10A, E, E, E, E, E, E, E, E, J, E, E, J, 1A, 1A, E, E, J, J, 1A, 5E, 5E]   
  3  SPZLINE       1 BinTableHDU     48   29R x 19C   [J, J, J, 13A, D, E, E, E, E, E, E, E, E, E, E, J, J, E, E]   
  4  B2-00003960-00003957-00003958    1 BinTableHDU    147   2047R x 7C   [E, E, E, J, E, E, E]   
  5  B2-00003961-00003957-00003958    1 BinTa

In [7]:
# Extraer la tabla de datos de la extensión COADD
spec_data = spec_hdulist[1].data  
flux = spec_data['flux']      # flujo en unidades de 1e-17 erg/s/cm^2/Å
loglam = spec_data['loglam']  # log10(lambda [Å])
ivar = spec_data['ivar']      # "inverse variance" del flujo

# Convertir log-lambda a lambda (Å)
wavelength = 10**loglam  # array de longitudes de onda en Angstroms

# Calcular incertidumbre a partir de la varianza inversa
flux_unc = np.zeros_like(flux)
positive = ivar > 0
flux_unc[positive] = 1/np.sqrt(ivar[positive])  # (en las mismas unidades que flux)

print(f"Espectro SDSS cargado: {len(wavelength)} píxeles de {wavelength.min():.1f}Å a {wavelength.max():.1f}Å")

Espectro SDSS cargado: 3817 píxeles de 3831.8Å a 9225.7Å


In [9]:
# Constante: flujo de 0 mag AB en cgs
f0 = 3.631e-20  # erg/s/cm^2/Hz

# Convertir f_lambda (cgs) a f_nu (cgs) y luego a maggies
# Nuestro flux actualmente está en 1e-17 cgs, así que multiplique por 1e-17 para tener erg/s/cm^2/Å:
flux_cgs = flux * 1e-17   
unc_cgs = flux_unc * 1e-17

# Convertir a f_nu: f_nu = f_lambda * (lambda^2 / c)
lam_cm = wavelength * 1e-8  # Angstrom a cm
c_cgs = c.to('cm/s').value  # velocidad de la luz en cm/s
f_nu = flux_cgs * (lam_cm**2 / c_cgs)
f_nu_unc = unc_cgs * (lam_cm**2 / c_cgs)

# Ahora convertir a maggies
flux_maggies = f_nu / f0
unc_maggies = f_nu_unc / f0

# Creamos el diccionario de observación para Prospector:
obs = {
    'wavelength': wavelength,
    'spectrum': flux_maggies,
    'unc': unc_maggies,
    'mask': np.ones_like(flux_maggies, dtype=bool)  # máscara: usar todos los datos por defecto
}

In [20]:
import fsps
from prospect.models.templates import TemplateLibrary
from prospect.models import SpecModel
from prospect.models import priors
from prospect.sources import CSPSpecBasis, FastStepBasis

In [21]:
# 1) Asegura FSPS y variables de entorno
import os
os.environ["SPS_HOME"] = "/home/hollman/fsps/"
os.environ["LD_LIBRARY_PATH"] = "/home/hollman/fsps/src:" + os.environ.get("LD_LIBRARY_PATH", "")

import fsps  # <- imprescindible tenerlo importado aquí

# 2) Inyecta fsps en los módulos de Prospector que lo usan sin importarlo
import importlib
import prospect.sources.ssp_basis as sspb        # FastStepBasis vive aquí
sspb.fsps = fsps
importlib.reload(sspb)                           # recarga para que lo "vea"

# (Opcional) si también usas CSPSpecBasis en el mismo kernel:
import prospect.sources.galaxy_basis as gb
gb.fsps = fsps
importlib.reload(gb)

# 3) Ahora sí, crea el SPS
from prospect.sources import FastStepBasis, CSPSpecBasis

In [22]:
# Definir número de bins de edad y sus límites (log10(años)):
nbins = 5
agebins = np.array([
    [np.log10(1e6),  np.log10(3e7)],    # 0 - 30 Myr
    [np.log10(3e7),  np.log10(3e8)],    # 30 - 300 Myr
    [np.log10(3e8),  np.log10(1.5e9)],  # 0.3 - 1.5 Gyr
    [np.log10(1.5e9),np.log10(5e9)],    # 1.5 - 5 Gyr
    [np.log10(5e9),  np.log10(13.6e9)]  # 5 - 13.6 Gyr 
])

# Cargar la plantilla de SFH de continuidad y luego ajustarla:
model_params = TemplateLibrary["continuity_sfh"]

# Modificar los parámetros de la SFH para nbins:
model_params["agebins"]["N"] = nbins
model_params["agebins"]["init"] = agebins  # establecer nuestros límites de bin definidos

model_params["logsfr_ratios"]["N"] = nbins - 1  # habrá nbins-1 razones entre bins
model_params["logsfr_ratios"]["init"] = [0.0] * (nbins - 1)  # inicializar asumiendo SFR ~ constante (razones=1 -> log10=0)
# Prior Student-t (df=2) en log SFR ratios para continuidad (evita cambios bruscos):
model_params["logsfr_ratios"]["prior"] = priors.StudentT(mean=np.zeros(nbins-1), scale=np.ones(nbins-1)*0.3, df=2)

# Asegurarnos de que logmass esté definido y libre:
if "logmass" in model_params:
    model_params["logmass"]["isfree"] = True
    model_params["logmass"]["init"] = 10.0  # valor inicial (masa total ~1e10 M_sun, arbitrario)
    model_params["logmass"]["prior"] = priors.TopHat(mini=7.0, maxi=12.0)  # prior uniforme en masa total (1e7 a 1e12 M_sun)
else:
    model_params["logmass"] = {"N":1, "isfree": True, "init": 10.0,
                               "prior": priors.TopHat(mini=7.0, maxi=12.0)}

# Añadir metalicidad estelar (log(Z/Zsol)):
model_params["logzsol"] = {"N":1, "isfree": True, "init": -0.5,
                           "prior": priors.TopHat(mini=-1.5, maxi=0.19)}

# Añadir parámetro de polvo (dust2, ley Charlot & Fall):
model_params["dust2"] = {"N":1, "isfree": True, "init": 0.2,
                         "prior": priors.TopHat(mini=0.0, maxi=1.0)}

# Fijar redshift conocido:
model_params["zred"] = {"N":1, "isfree": False, "init": 0.02246}

# Incorporar parámetros nebulares:
model_params.update(TemplateLibrary["nebular"])
# (Esto activa add_neb_emission=True, add_neb_continuum=True, nebemlineinspec=False,
#  amarra gas_logz = logzsol, e incluye gas_logu libre con prior)

# Construir el modelo de espectro y el objeto SPS (FSPS) apropiado:
model = SpecModel(model_params)
sps = FastStepBasis(zcontinuous=1)  # usamos FastStepBasis para SFH no paramétrica (bins)

In [23]:
print("Parámetros libres:", model.free_params)

Parámetros libres: ['logzsol', 'dust2', 'logmass', 'logsfr_ratios']


In [24]:
from prospect.fitting import fit_model, lnprobfn

# Configurar parámetros de muestreo (valores reducidos para demostración):
fit_kwargs = dict(
    optimize=True,
    dynesty=True,
    nlive_init=100,         # número de live points inicial (pequeño para demo; use >300 en aplicaciones reales)
    nested_method='rwalk',  # método de muestreo (rwalk = random walk)
    nested_dlogz_init=0.5   # criterio de parada (log-evidencia), aquí relajado para velocidad
)

# Ejecutar el ajuste:
output = fit_model(obs, model, sps, lnprobfn=lnprobfn, **fit_kwargs)

# Extraer resultados del muestreo:
sampling_results, sampling_duration = output["sampling"]
print(f"Muestreo completado en {sampling_duration:.1f} segundos")

/home/hollman/miniconda3/envs/prospector/lib/python3.10/site-packages/prospect/models/priors.py:125: RuntimeWarning: divide by zero encountered in log
  lnp = np.log(p)
iter: 5653 | batch: 0 | nc: 2421 | ncall: 520580 | eff(%):  1.086 | logz: -36817012411000392.000 +/-    nan | dlogz:    inf >  0.500     0           

There was an error during the likelihood call at parameters [ 0.18999854  0.99999924  7.00000145  0.56622451  0.03211357  0.13873167
 -8.61042747]
Exception while calling loglikelihood function:
  params: [ 0.18999854  0.99999924  7.00000145  0.56622451  0.03211357  0.13873167
 -8.61042747]
  args: []
  kwargs: {}
  exception:


Traceback (most recent call last):
  File "/home/hollman/miniconda3/envs/prospector/lib/python3.10/site-packages/dynesty/dynesty.py", line 913, in __call__
    return self.func(np.asarray(x).copy(), *self.args, **self.kwargs)
  File "/home/hollman/miniconda3/envs/prospector/lib/python3.10/site-packages/prospect/fitting/fitting.py", line 114, in lnprobfn
    spec, phot, x = model.predict(theta, obs, sps=sps, sigma_spec=sigma_spec)
  File "/home/hollman/miniconda3/envs/prospector/lib/python3.10/site-packages/prospect/models/sedmodel.py", line 101, in predict
    self._wave, self._spec, self._mfrac = sps.get_galaxy_spectrum(**self.params)
  File "/home/hollman/miniconda3/envs/prospector/lib/python3.10/site-packages/prospect/sources/ssp_basis.py", line 344, in get_galaxy_spectrum
    wave, spec = self.ssp.get_spectrum(tage=tmax, peraa=False)
  File "/home/hollman/miniconda3/envs/prospector/lib/python3.10/site-packages/fsps/fsps.py", line 609, in get_spectrum
    self._compute_csp()
  File 

KeyboardInterrupt: 

In [25]:
from prospect.plotting.utils import best_sample

# Obtener la muestra de mayor posterior:
theta_maxpost = best_sample(sampling_results)
print("Parámetros del mejor ajuste:")
for name, val in zip(model.theta_labels(), theta_maxpost):
    print(f"  {name} = {val:.3f}")

NameError: name 'sampling_results' is not defined

In [ ]:
# Obtener el espectro modelado y las fracciones de masa (mfrac) del mejor ajuste
spec_model, phot_model, mfrac = model.predict(theta_maxpost, obs=obs, sps=sps)

print("Fracción de masa por bin de edad:")
for i, frac in enumerate(mfrac):
    age0, age1 = 10**agebins[i][0], 10**agebins[i][1]  # límites del bin en años
    label = f"{age0/1e6:.0f}-{age1/1e6:.0f} Myr" if age1 < 1e9 else f"{age0/1e9:.1f}-{age1/1e9:.1f} Gyr"
    print(f"  Bin {i+1} ({label}): {frac*100:.1f}% de la masa estelar")

# Calcular fracción de luz en banda V (~5000-6000 Å) para cada bin:
wave_model = sps.wavelengths  # longitud de onda del modelo (rest-frame)
v_band_range = (5000, 6000)   # definimos banda V aproximada como 5000-6000 Å
total_lum_v = 0.0
lum_v_per_bin = []

# Inicializar FSPS para generar espectro de cada bin individualmente
sp = fsps.StellarPopulation(zcontinuous=1)
# Configurar los parámetros globales de FSPS (metalicidad y polvo igual al mejor ajuste, para consistencia)
sp.params['logzsol'] = theta_maxpost[model.theta_index['logzsol']]
sp.params['dust2']   = theta_maxpost[model.theta_index['dust2']]
sp.params['dust1']   = sp.params['dust2']  # aplicar la misma atenuación en regiones de nacimiento estelar (simplificación)
sp.params['add_neb_emission'] = True
sp.params['add_neb_continuum'] = True
sp.params['gas_logu'] = theta_maxpost[model.theta_index['gas_logu']]
sp.params['gas_logz'] = theta_maxpost[model.theta_index['logzsol']]  # mismo Z para gas

# Establecer SFH tabular (sfh=3) en FSPS
sp.params['sfh'] = 3
sp.params['tage'] = 13.7  # edad del Universo aproximada en Gyr (para cerrar SFH)
# Iterar sobre cada bin de edad:
for i, frac in enumerate(mfrac):
    # Calcular masa formada en este bin (en unidades de M_sun)
    total_mass_formed = 10**theta_maxpost[model.theta_index['logmass']]  # masa total formada (M☉)
    mass_i = frac * total_mass_formed
    
    # Determinar tiempos de formación (t_start, t_end) en Gyr desde Big Bang:
    # (Lookback agebins están desde presente; convertimos a tiempo absoluto desde t=0)
    t_start_lookback = 10**agebins[i][1] / 1e9  # Gyr
    t_end_lookback   = 10**agebins[i][0] / 1e9  # Gyr
    age_univ = 13.7  # (aprox) edad del universo en Gyr
    t_form_start = max(age_univ - t_end_lookback, 0)
    t_form_end   = max(age_univ - t_start_lookback, 0)
    # (Por ejemplo, si bin es 5-13.6 Gyr lookback, t_form_start ~ 0.1 Gyr, t_form_end ~8.7 Gyr,
    #  lo que significa que la formación ocurrió entre 0.1 y 8.7 Gyr después del Big Bang.)
    
    # Calcular una tasa de formación constante en ese intervalo para producir mass_i:
    delta_t = t_form_end - t_form_start  # duración del bin en Gyr
    sfr = mass_i / (delta_t * 1e9)  # M☉/yr constante en ese intervalo
    
    # Configurar la SFH tabular en FSPS para este bin:
    sp.set_tabular_sfh(age=np.array([t_form_start, t_form_end]), sfr=np.array([sfr, sfr]))
    
    # Obtener espectro de este bin:
    w, spec_lum = sp.get_spectrum(tage=sp.params['tage'], peraa=True)
    # spec_lum está en L_sun/Å (luminosidad espectral). Convertimos a erg/s/Å:
    spec_lum_erg = spec_lum * fsps.constants.L_sun  # L_sun ~3.828e33 erg/s
    
    # Integrar luminosidad en el rango V (5000-6000 Å):
    in_band = (w >= v_band_range[0]) & (w <= v_band_range[1])
    lum_v = np.trapz(spec_lum_erg[in_band], w[in_band])  # erg/s en banda V (aprox)
    lum_v_per_bin.append(lum_v)
    total_lum_v += lum_v

# Convertir a fracciones de luz:
light_fraction = np.array(lum_v_per_bin) / total_lum_v

print("\nFracción de luz (banda ~V) por bin de edad:")
for i, frac in enumerate(light_fraction):
    age0, age1 = 10**agebins[i][0], 10**agebins[i][1]
    label = f"{age0/1e6:.0f}-{age1/1e6:.0f} Myr" if age1 < 1e9 else f"{age0/1e9:.1f}-{age1/1e9:.1f} Gyr"
    print(f"  Bin {i+1} ({label}): {frac*100:.1f}% de la luz V")

In [ ]:
import matplotlib.pyplot as plt

# Definir etiquetas de cada bin en términos de edad (para el eje X):
age_labels = ["0-30 Myr", "30-300 Myr", "0.3-1.5 Gyr", "1.5-5 Gyr", ">5 Gyr"]

# Gráfico 1: Espectro observado vs modelo
plt.figure(figsize=(8,4))
mask_plot = (obs['wavelength'] > 3600) & (obs['wavelength'] < 7000)  # rango visual
plt.plot(obs['wavelength'][mask_plot], obs['spectrum'][mask_plot], 
         color='gray', alpha=0.7, label='Espectro observado')
plt.plot(obs['wavelength'][mask_plot], spec_model[mask_plot], 
         color='red', label='Modelo (ajuste)')
plt.xlabel("Longitud de onda (Å)")
plt.ylabel("Flujo (maggies)")
plt.title("Espectro observado vs. modelo ajustado")
plt.legend()
plt.show()

# Gráfico 2: Barras de fracción de masa vs edad
plt.figure(figsize=(6,4))
plt.bar(range(nbins), mfrac*100, color='skyblue')
plt.xticks(range(nbins), age_labels, rotation=45)
plt.ylabel("Fracción de masa (%)")
plt.title("Distribución de masa estelar por rango de edad")
plt.show()

# Gráfico 3: Barras de fracción de luz vs edad
plt.figure(figsize=(6,4))
plt.bar(range(nbins), light_fraction*100, color='orange')
plt.xticks(range(nbins), age_labels, rotation=45)
plt.ylabel("Fracción de luz óptica (%)")
plt.title("Distribución de luz (banda V) por rango de edad")
plt.show()